In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import numpy as np

2025-10-05 19:08:24.349323: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
#teacher model (simpler model)
teachers_model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
teachers_model.compile(optimizer=optimizers.Adam(),
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])

#Assume the teacher model is already trained
#for demonstration, we will skip training
#student model (smaller model)
student_model = models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(10, activation='softmax')
])
student_model.compile(optimizer=optimizers.Adam(),
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

# Fix data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, y_train = x_train[:1000], y_train[:1000]
x_train = (x_train.astype('float32') / 255.0)[..., None]  # add channel
x_test  = (x_test.astype('float32') / 255.0)[..., None]

# (Quick) train teacher a bit so it has signal
teachers_model.fit(x_train, y_train, epochs=2, batch_size=64, verbose=0)

def distillation_loss(teacher_probs, student_probs):
    # KL(teacher || student) with probs (avoid double softmax)
    return tf.reduce_mean(
        tf.reduce_sum(
            teacher_probs * (tf.math.log(teacher_probs + 1e-8) - tf.math.log(student_probs + 1e-8)),
            axis=1
        )
    )

def train_student(student, teacher, x_train, y_train, epochs=3, batch_size=32, alpha=0.5):
    ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(1000).batch(batch_size)
    for epoch in range(epochs):
        for x_batch, y_batch in ds:
            teacher_probs = teacher(x_batch, training=False)          # already softmax
            with tf.GradientTape() as tape:
                student_probs = student(x_batch, training=True)
                soft = distillation_loss(teacher_probs, student_probs)
                hard = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(y_batch, student_probs)
                )
                loss = alpha * hard + (1 - alpha) * soft
            grads = tape.gradient(loss, student.trainable_variables)
            student.optimizer.apply_gradients(zip(grads, student.trainable_variables))
        print(f"Epoch {epoch+1}/{epochs} - loss: {loss.numpy():.4f}")

train_student(student_model, teachers_model, x_train, y_train, epochs=3)
student_model.evaluate(x_test, y_test, verbose=1)

2025-10-05 19:34:12.084951: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1/3 - loss: 0.8831


2025-10-05 19:34:12.716515: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 2/3 - loss: 0.2757
Epoch 3/3 - loss: 0.3309


2025-10-05 19:34:13.367674: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 31360000 exceeds 10% of free system memory.


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7407 - loss: 0.8722


[0.7691925764083862, 0.777400016784668]